# Course 5 — Ensembles

Bagging, random forests, and gradient boosting. The algorithms that
win Kaggle competitions and ship in production at every data-driven
company.

**Sections**
1. Random forests on Carseats (0:00–0:30)
2. OOB error, feature importance, partial dependence (0:30–1:00)
3. Gradient boosting and the learning-rate dial (1:00–1:30)

In [ ]:
import sys, pathlib
p = pathlib.Path.cwd()
while not (p / 'pyproject.toml').exists() and p != p.parent:
    p = p.parent
if str(p) not in sys.path:
    sys.path.insert(0, str(p))
from shared.data_utils import load_dataset
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import time
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor, GradientBoostingClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.inspection import PartialDependenceDisplay
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import accuracy_score, mean_squared_error, roc_auc_score
carseats = load_dataset('carseats')
boston = load_dataset('boston')


## 1. Random forests

A random forest fits many trees, each on a bootstrap sample, each
considering only a random subset of features at every split.
`max_features='sqrt'` is the classic default for classification.

In [ ]:
c = carseats.copy()
c['High'] = (c['Sales'] > 8).astype(int)
c = pd.get_dummies(c.drop(columns=['Sales']), columns=['ShelveLoc','Urban','US'],
                    drop_first=True, dtype=float)
X = c.drop(columns=['High']); y = c['High']
Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.3, random_state=0, stratify=y)

tree = DecisionTreeClassifier(random_state=0).fit(Xtr, ytr)
rf = RandomForestClassifier(n_estimators=300, max_features='sqrt',
                             oob_score=True, n_jobs=-1, random_state=0).fit(Xtr, ytr)
print(f'single tree test acc = {tree.score(Xte, yte):.4f}')
print(f'random forest test  = {rf.score(Xte, yte):.4f}')
print(f'OOB error           = {1 - rf.oob_score_:.4f}')


## 2. OOB, feature importance, partial dependence

In [ ]:
# Regression forest on Boston for the feature-importance story
Xb = boston.drop(columns=['medv']); yb = boston['medv']
rfr = RandomForestRegressor(n_estimators=300, n_jobs=-1, random_state=0).fit(Xb, yb)
imp = pd.Series(rfr.feature_importances_, index=Xb.columns).sort_values()
fig, ax = plt.subplots(figsize=(6, 4))
imp.plot.barh(ax=ax); ax.set_title('RF feature importance on Boston')
ax.set_xlabel('importance'); plt.show()


In [ ]:
top2 = imp.tail(2).index.tolist()
fig, ax = plt.subplots(figsize=(10, 4))
PartialDependenceDisplay.from_estimator(rfr, Xb, top2, ax=ax)
plt.show()


## 3. Gradient boosting

Boosting fits trees *sequentially*, each one predicting the errors of
the previous ensemble. The learning rate controls how much each tree
contributes — small rate + many trees almost always beats big rate + few.

In [ ]:
results = []
for lr in [0.01, 0.1]:
    for n in [100, 500]:
        t0 = time.perf_counter()
        m = GradientBoostingClassifier(learning_rate=lr, n_estimators=n,
                                        max_depth=3, random_state=0).fit(Xtr, ytr)
        acc = m.score(Xte, yte)
        results.append((lr, n, acc, time.perf_counter() - t0))
        print(f'lr={lr:.2f}  n={n:4d}  acc={acc:.4f}  t={results[-1][3]:.2f}s')


`HistGradientBoostingClassifier` is the modern fast default — same
math, histogram-based splits, often 10–100× faster on large data.

### Recap
- Random forests = bagging + random feature subsets. Decorrelate the trees.
- OOB error is a free CV. Feature importance and PDPs give interpretability back.
- Gradient boosting fits residuals sequentially. Tune learning_rate × n_estimators.
- For production tabular work today, start with GBM/XGBoost/LightGBM.

---

# Exercises

Each exercise below is followed by its solution.
Try to solve the tasks yourself before revealing the next cell.

---

## Exercise 1

**Task 1.** Reload `tips`, build `high_tip = (tip/total_bill) > 0.18`,
and fit three models on the same train/test split: single decision tree,
random forest (300 trees), gradient boosting (300 trees). Report
test accuracy for each.

In [ ]:
import sys, pathlib
p = pathlib.Path.cwd()
while not (p / 'pyproject.toml').exists() and p != p.parent:
    p = p.parent
if str(p) not in sys.path:
    sys.path.insert(0, str(p))
from shared.data_utils import load_dataset
import pandas as pd
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.model_selection import train_test_split
# your code here


### Exercise 1 — Solution

In [ ]:
import sys, pathlib
p = pathlib.Path.cwd()
while not (p / 'pyproject.toml').exists() and p != p.parent:
    p = p.parent
if str(p) not in sys.path:
    sys.path.insert(0, str(p))
from shared.data_utils import load_dataset
import pandas as pd
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.model_selection import train_test_split
tips = load_dataset('tips')
tips['high_tip'] = ((tips['tip']/tips['total_bill']) > 0.18).astype(int)
X = pd.get_dummies(tips[['total_bill','size','day','time','sex','smoker']],
                    columns=['day','time','sex','smoker'], drop_first=True, dtype=float)
y = tips['high_tip']
Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.3, random_state=0, stratify=y)
tree = DecisionTreeClassifier(random_state=0).fit(Xtr, ytr)
rf = RandomForestClassifier(n_estimators=300, n_jobs=-1, random_state=0).fit(Xtr, ytr)
gbm = GradientBoostingClassifier(n_estimators=300, random_state=0).fit(Xtr, ytr)
for name, m in [('tree', tree), ('rf', rf), ('gbm', gbm)]:
    print(f'{name:5s}: acc = {m.score(Xte, yte):.4f}')


---

## Exercise 2

**Task 2.** Compare AUC of the three models.

In [ ]:
# your code here


### Exercise 2 — Solution

In [ ]:
from sklearn.metrics import roc_auc_score
for name, m in [('tree', tree), ('rf', rf), ('gbm', gbm)]:
    p = m.predict_proba(Xte)[:, 1]
    print(f'{name:5s}: AUC = {roc_auc_score(yte, p):.4f}')


---

## Exercise 3

**Task 3.** Plot the top-5 feature importances from the random forest.

In [ ]:
# your code here


### Exercise 3 — Solution

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
imp = pd.Series(rf.feature_importances_, index=X.columns).sort_values().tail(5)
fig, ax = plt.subplots(figsize=(6, 3))
imp.plot.barh(ax=ax); ax.set_xlabel('importance'); plt.show()
